# NVIDIA Developer Segmentation — Clustering Pipeline
**UMAP + HDBSCAN on multi-dimensional engagement features**

Reads from `developer_project.duckdb` using cleaned tables from `Cleaning.ipynb`.

Input tables: `activity_final`, `contact_final`

Output: `dev_segments_v1` written back to DuckDB

Pipeline:
1. Pull and engineer features from DuckDB
2. Normalize and reduce with UMAP
3. Cluster with HDBSCAN
4. Profile clusters
5. Write segment labels back to DuckDB

## 0. Imports & Config

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import hdbscan
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)

con = duckdb.connect('developer_project.duckdb')
RANDOM_STATE = 42

print('Connected. Tables available:')
display(con.execute('SHOW TABLES').fetchdf())

## 1. Feature Engineering

Instead of using the raw `activity_score` as a single number, we split it into **6 latent axes** that capture distinct developer behaviors. All activity labels are already lowercased in `activity_final`.

| Feature | Activities included | What it captures |
|---|---|---|
| `learn_score` | dli training, on-demand views | Structured learning |
| `build_score` | devzone downloads, ngc downloads, hosted api, brev | Active building |
| `community_score` | forum contributions, hackathon, conference, other events, conf sessions live | Community presence |
| `visibility_score` | webinars, user feedback, contests, program applications, dev program membership | Awareness / programs |
| `breadth_score` | count of distinct activity types | Platform breadth |
| `recency_score` | days since last activity (inverted) | How recently active |

In [ ]:
feature_query = """
WITH activity_axes AS (
    SELECT
        dev_contact AS developer_id,

        SUM(CASE WHEN activity IN ('dli training', 'on-demand views')
            THEN activity_score ELSE 0 END)                              AS learn_score,

        SUM(CASE WHEN activity IN ('devzone downloads', 'ngc downloads', 'hosted api', 'brev')
            THEN activity_score ELSE 0 END)                              AS build_score,

        SUM(CASE WHEN activity IN ('forum contributions', 'hackathon', 'conference',
                                   'other events', 'conf sessions live')
            THEN activity_score ELSE 0 END)                              AS community_score,

        SUM(CASE WHEN activity IN ('webinars', 'user feedback', 'contests',
                                   'program applications', 'dev program membership')
            THEN activity_score ELSE 0 END)                              AS visibility_score,

        SUM(CASE WHEN activity_role LIKE '%speaker%'
                   OR activity_role LIKE '%presenter%'
            THEN 1 ELSE 0 END)                                           AS speaker_count,

        COUNT(DISTINCT activity)                                          AS breadth_score,
        COUNT(*)                                                          AS total_activities,
        MAX(activity_date)                                                AS last_activity_date,
        MIN(activity_date)                                                AS first_activity_date
    FROM activity_final
    GROUP BY dev_contact
),

contact_dim AS (
    SELECT
        developer_id,
        created_date,
        country,
        region,
        account_type,
        inception_id,
        industry_segment_vertical,
        development_areas,
        fields_of_interest,
        wwfo_category,
        account_id,
        normalized_account_name
    FROM contact_final
)

SELECT
    a.*,
    c.created_date,
    c.country,
    c.region,
    c.account_type,
    c.inception_id,
    c.industry_segment_vertical,
    c.development_areas,
    c.fields_of_interest,
    c.wwfo_category,
    c.account_id,
    c.normalized_account_name,
    DATEDIFF('day', c.created_date, CURRENT_DATE)      AS tenure_days,
    DATEDIFF('day', a.last_activity_date, CURRENT_DATE) AS days_since_last_activity
FROM activity_axes a
LEFT JOIN contact_dim c ON a.developer_id = c.developer_id
"""

df = con.execute(feature_query).fetchdf()
print(f'Feature dataframe: {df.shape}')
display(df.head(3))

In [ ]:
# Derived features
df['days_since_last_activity'] = df['days_since_last_activity'].fillna(9999)
df['recency_score']  = 1 / (1 + df['days_since_last_activity'])
df['tenure_days']    = df['tenure_days'].fillna(0).clip(lower=0)
df['learn_per_day']  = df['learn_score']  / (df['tenure_days'] + 1)
df['build_per_day']  = df['build_score']  / (df['tenure_days'] + 1)
df['is_inception']   = df['inception_id'].notna().astype(int)

print('Derived features added.')
display(df[['learn_score','build_score','community_score','visibility_score',
            'breadth_score','recency_score','speaker_count',
            'tenure_days','learn_per_day','build_per_day']].describe())

## 2. Prepare Feature Matrix

In [ ]:
FEATURE_COLS = [
    'learn_score', 'build_score', 'community_score', 'visibility_score',
    'breadth_score', 'recency_score', 'speaker_count',
    'tenure_days', 'learn_per_day', 'build_per_day', 'is_inception'
]

X = df[FEATURE_COLS].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

print(f'Feature matrix: {X_scaled.shape}')

## 3. UMAP Dimensionality Reduction

Key params to tune:
- `n_neighbors`: increase to 30 for smoother global structure
- `min_dist=0.0`: keeps points tightly packed — better for HDBSCAN
- `n_components=2`: 2D for visualization; use 5-10 for a production-only pipeline

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=2,
    metric='euclidean',
    random_state=RANDOM_STATE
)

print('Running UMAP... (may take a minute on large data)')
embedding = reducer.fit_transform(X_scaled)

df['umap_x'] = embedding[:, 0]
df['umap_y'] = embedding[:, 1]
print('UMAP complete.')

## 4. HDBSCAN Clustering

Key params to tune:
- `min_cluster_size`: raise if you want fewer, coarser clusters
- `min_samples`: higher = more conservative = more noise points
- `cluster_selection_method='eom'`: excess-of-mass — finds stable clusters

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(embedding)
df['cluster'] = cluster_labels
df['cluster_probability'] = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print(f'Clusters found : {n_clusters}')
print(f'Noise points   : {n_noise} ({100 * n_noise / len(cluster_labels):.1f}%)')
print('\nCluster sizes:')
display(pd.Series(cluster_labels).value_counts().sort_index()
        .rename('count').rename_axis('cluster').reset_index())

## 5. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

noise = df[df['cluster'] == -1]
ax.scatter(noise['umap_x'], noise['umap_y'],
           c='lightgrey', s=5, alpha=0.4, label='Noise (-1)')

clustered = df[df['cluster'] >= 0]
palette   = sns.color_palette('tab10', n_colors=n_clusters)

for i, cid in enumerate(sorted(clustered['cluster'].unique())):
    mask = clustered['cluster'] == cid
    ax.scatter(
        clustered.loc[mask, 'umap_x'],
        clustered.loc[mask, 'umap_y'],
        s=15, alpha=0.7, color=palette[i],
        label=f'Cluster {cid} (n={mask.sum():,})'
    )

ax.set_title('NVIDIA Developer Segments — UMAP + HDBSCAN', fontsize=14, fontweight='bold')
ax.set_xlabel('UMAP Dimension 1')
ax.set_ylabel('UMAP Dimension 2')
ax.legend(markerscale=2, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('umap_clusters.png', dpi=150)
plt.show()

## 6. Cluster Profiles

In [ ]:
profile_cols = [
    'learn_score', 'build_score', 'community_score', 'visibility_score',
    'breadth_score', 'recency_score', 'speaker_count',
    'tenure_days', 'total_activities'
]

clustered_df = df[df['cluster'] >= 0]

cluster_profiles = clustered_df.groupby('cluster')[profile_cols].mean().round(2)
cluster_profiles['n_developers'] = clustered_df.groupby('cluster')['developer_id'].count()

print('Cluster mean profiles:')
display(cluster_profiles)

In [ ]:
# Heatmap: normalized colors, raw means as annotations
profile_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(cluster_profiles[profile_cols]),
    index=cluster_profiles.index,
    columns=profile_cols
)

fig, ax = plt.subplots(figsize=(12, max(4, n_clusters)))
sns.heatmap(
    profile_norm,
    annot=cluster_profiles[profile_cols].round(1),
    fmt='g', cmap='YlOrRd', linewidths=0.5, ax=ax
)
ax.set_title('Cluster Profiles — Normalized (color) | Raw Mean (annotation)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_profiles_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Top industries and regions per cluster
for cid in sorted(clustered_df['cluster'].unique()):
    sub = clustered_df[clustered_df['cluster'] == cid]
    print(f'\n=== Cluster {cid} (n={len(sub):,}) ===')
    print('  Top industries:')
    display(sub['industry_segment_vertical'].value_counts().head(5))
    print('  Top regions:')
    display(sub['region'].value_counts().head(5))
    print('  Top development areas:')
    display(sub['development_areas'].value_counts().head(5))

## 7. Assign Segment Labels

After reviewing the profiles above, update the dictionary below with descriptive names.
Expected NVIDIA archetypes: `cuda_optimizer`, `genai_builder`, `community_champion`, `passive_learner`, `robotics_edge`

In [ ]:
# Update after reviewing cluster profiles
SEGMENT_LABELS = {
    -1: 'noise',
    0:  'relabel_me',
    1:  'relabel_me',
    2:  'relabel_me',
    3:  'relabel_me',
    4:  'relabel_me',
}

df['segment_label'] = df['cluster'].map(SEGMENT_LABELS).fillna('relabel_me')

print('Segment distribution:')
display(df['segment_label'].value_counts())

## 8. Write Results to DuckDB

In [ ]:
output_cols = [
    'developer_id', 'cluster', 'cluster_probability', 'segment_label',
    'learn_score', 'build_score', 'community_score', 'visibility_score',
    'breadth_score', 'recency_score', 'speaker_count',
    'tenure_days', 'total_activities', 'learn_per_day', 'build_per_day',
    'is_inception', 'umap_x', 'umap_y',
    'last_activity_date', 'country', 'region', 'account_type',
    'industry_segment_vertical', 'development_areas',
    'account_id', 'normalized_account_name'
]

segments_df = df[output_cols].copy()

con.execute('DROP TABLE IF EXISTS dev_segments_v1')
con.execute('CREATE TABLE dev_segments_v1 AS SELECT * FROM segments_df')

print('dev_segments_v1 written to DuckDB.')
display(con.execute(
    'SELECT segment_label, COUNT(*) AS n FROM dev_segments_v1 GROUP BY segment_label ORDER BY n DESC'
).fetchdf())

## 9. Coverage Check

In [ ]:
total    = con.execute('SELECT COUNT(DISTINCT dev_contact) FROM activity_final').fetchone()[0]
segmented = con.execute('SELECT COUNT(*) FROM dev_segments_v1').fetchone()[0]

print(f'Developers in activity_final : {total:,}')
print(f'Developers in dev_segments_v1: {segmented:,}')
print(f'Coverage                     : {100 * segmented / total:.1f}%')

display(con.execute('SELECT * FROM dev_segments_v1 LIMIT 5').fetchdf())